# Import Libraries

In [239]:
import pandas as pd
import numpy as np
import spacy
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Input, Dropout, GlobalMaxPooling1D, Conv1D, Bidirectional, BatchNormalization, SimpleRNN, Attention, GlobalAveragePooling1D, Bidirectional
from tensorflow.keras.optimizers import Adam
from gensim.models import Word2Vec
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [240]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

In [292]:
# Tokenize input text
# Load BERT tokenizer and model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = TFBertModel.from_pretrained(model_name)

def tokenize_texts(texts, max_len):
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='tf'
    )

    outputs = bert_model(encodings)

    return outputs.last_hidden_state

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [242]:
def make_dataset(texts, labels, tokenizer, shuffle=False, max_len= 20, batch_size = 32):
    def gen():
        for t, l in zip(texts, labels):
            enc = tokenizer(
                t,
                truncation=True,
                padding='max_length',
                max_length=max_len,
                return_tensors='tf'
            )
            yield ({'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0]}, l)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'input_ids': tf.TensorSpec(shape=(max_len,), dtype=tf.int32),
             'attention_mask': tf.TensorSpec(shape=(max_len,), dtype=tf.int32)},
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Pre-Processing

### Import Data

In [312]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
df = pd.DataFrame()
for i in [2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

max_len = max(len(tokenizer.encode(text, add_special_tokens=True)) for text in df['question'])
print("Max sequence length:", max_len)

Max sequence length: 95


In [313]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

test_df['processed_question'] = test_df['question'].apply(preprocess_text)
test_df['processed_question'] = [''.join(text) for text in test_df['processed_question']]

## Tokenize
### Execute any one

### BERT

In [314]:
# Parameters
batch_size = 32
num_classes = 6

# Embedding
x_train = tokenize_texts(df['question'], max_len)
x_test = tokenize_texts(test_df['question'], max_len)

In [273]:
# test embedding
input_ids, attention_mask = tokenize_texts(test_df['question'], max_len)

bert_outputs = bert_model(input_ids, attention_mask=attention_mask)
test_embeddings = bert_outputs.last_hidden_state  # shape (batch_size, max_len, 768)

In [315]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = df['label'].map(y_mapper)
y_test_mapped = test_df['label'].map(y_mapper)

y_train = to_categorical(np.asarray(y_mapped))
y_test = to_categorical(np.asarray(y_test_mapped))

# Modelling

## 1D CNN

In [316]:
cnn_model = Sequential([
    # Input(shape=(max_len, 768)),

    Conv1D(128, 5, activation='gelu', padding= 'same', input_shape=(max_len, 768)),
    BatchNormalization(),
    GlobalMaxPooling1D(),
    Dropout(0.3),

    Dense(64, activation='sigmoid'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

cnn_model.summary()

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_53"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_27 (Conv1D)              │ (None, 95, 128)        │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_27         │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_121 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_118 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_122 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_119 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,806 (1.91 MB)

 Trainable params: 500,550 (1.91 MB)

 Non-trainable params: 256 (1.00 KB)

In [317]:
cnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split= 0.2)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.1689 - loss: 2.2212 - val_accuracy: 0.1195 - val_loss: 1.9084
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3333 - loss: 1.6796 - val_accuracy: 0.4208 - val_loss: 1.6098
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.3995 - loss: 1.5182 - val_accuracy: 0.6104 - val_loss: 1.4323
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4329 - loss: 1.4692 - val_accuracy: 0.6779 - val_loss: 1.2580
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5035 - loss: 1.3473 - val_accuracy: 0.7143 - val_loss: 1.1184
Epoch 6/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5285 - loss: 1.2534 - val_accuracy: 0.7091 - val_loss: 1.0148
Epoch 7/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5791 - loss: 1.1319 - val_accuracy: 0.7117 - val_loss: 0.9524
Epoch 8/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6365 - loss: 1.1029 - val_accuracy: 0.

In [322]:
loss, acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 56.83%


## RNN

In [319]:
rnn_model = Sequential([
        Input(shape=(max_len, 768)),

        SimpleRNN(64, activation= 'tanh'),
        Dropout(0.3),
        
        Dense(32, activation='relu'),
        Dropout(0.4),
        
        Dense(6, activation='softmax')
    ])

rnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

rnn_model.summary()

Model: "sequential_54"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_8 (SimpleRNN)        │ (None, 64)             │        53,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_123 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_120 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_124 (Dropout)           │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_121 (Dense)               │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,590 (217.15 KB)

 Trainable params: 55,590 (217.15 KB)

 Non-trainable params: 0 (0.00 B)

In [320]:
rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.1544 - loss: 2.0401 - val_accuracy: 0.2883 - val_loss: 1.7115
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2374 - loss: 1.7912 - val_accuracy: 0.5013 - val_loss: 1.5634
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2830 - loss: 1.7422 - val_accuracy: 0.5636 - val_loss: 1.4993
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3014 - loss: 1.7298 - val_accuracy: 0.5740 - val_loss: 1.4678
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3552 - loss: 1.6444 - val_accuracy: 0.5662 - val_loss: 1.4529
Epoch 6/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3689 - loss: 1.6308 - val_accuracy: 0.5818 - val_loss: 1.4425
Epoch 7/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3781 - loss: 1.6224 - val_accuracy: 0.5481 - val_loss: 1.4401
Epoch 8/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3842 - loss: 1.6006 - val_accuracy: 0.

In [321]:
loss, acc = rnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 45.50%


## LSTM

In [308]:
lstm_model = Sequential([
    Input(shape=(max_len, 768)),

    # Stacked bidirectional LSTMs
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.25),

    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.25),

    # Dense classifier head
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.25),

    Dense(6, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

lstm_model.summary()

Model: "sequential_52"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_11                │ (None, 84, 256)        │       918,528 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_117 (Dropout)           │ (None, 84, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_12                │ (None, 128)            │       164,352 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_118 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_115 (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_119 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_116 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_120 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_117 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,108,038 (4.23 MB)

 Trainable params: 1,108,038 (4.23 MB)

 Non-trainable params: 0 (0.00 B)

In [310]:
lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - accuracy: 0.1952 - loss: 1.7820 - val_accuracy: 0.1811 - val_loss: 1.7607
Epoch 2/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - accuracy: 0.2724 - loss: 1.7212 - val_accuracy: 0.2792 - val_loss: 1.7161
Epoch 3/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/step - accuracy: 0.3468 - loss: 1.6736 - val_accuracy: 0.3509 - val_loss: 1.6690
Epoch 4/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 144ms/step - accuracy: 0.3788 - loss: 1.6203 - val_accuracy: 0.3849 - val_loss: 1.6029
Epoch 5/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 154ms/step - accuracy: 0.4349 - loss: 1.5382 - val_accuracy: 0.4151 - val_loss: 1.5292
Epoch 6/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 154ms/step - accuracy: 0.4726 - loss: 1.4447 - val_accuracy: 0.4415 - val_loss: 1.4144
Epoch 7/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 155ms/step - accuracy: 0.4986 - loss: 1.3700 - val_accuracy: 0.4717 - val_loss: 1.3526
Epoch 8/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 156ms/step - accuracy: 0.5838 - loss: 1.2138 - val_accura

KeyboardInterrupt: 

In [69]:
loss, acc = lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 21.67%


### Attention + LSTM

In [237]:
inputs = Input(shape=(max_len,768))

# 1. Embedding

# 2. LSTM layer with return_sequences=True
lstm_out = LSTM(64, activation='tanh', return_sequences=True)(inputs)

# 3. Self-attention: query=key=value from LSTM output
attn_out = Attention(use_scale=True)([lstm_out, lstm_out])

# 4. Flatten the attended sequence into a single vector
context = GlobalAveragePooling1D()(attn_out)

# 5. Dense layers
h = Dense(32, activation='relu')(context)
h = Dropout(0.4)(h)
outputs = Dense(6, activation='softmax')(h)

at_lstm_model = Model(inputs, outputs)
at_lstm_model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
at_lstm_model.summary()

Model: "functional_42"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_48      │ (None, 46, 768)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_23 (LSTM)      │ (None, 46, 64)    │    213,248 │ input_layer_48[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_2         │ (None, 46, 64)    │          1 │ lstm_23[0][0],    │
│ (Attention)         │                   │            │ lstm_23[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention_2[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_89 (Dense)    │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_90          │ (None, 32)        │          0 │ dense_89[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_90 (Dense)    │ (None, 6)         │        198 │ dropout_90[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 215,527 (841.90 KB)

 Trainable params: 215,527 (841.90 KB)

 Non-trainable params: 0 (0.00 B)

In [238]:
at_lstm_model.fit(train_embeddings, y_train, epochs=500, batch_size = 32, validation_split = 0.8)

Epoch 1/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.1558 - loss: 1.8697 - val_accuracy: 0.1705 - val_loss: 1.7948
Epoch 2/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.2335 - loss: 1.8262 - val_accuracy: 0.1684 - val_loss: 1.7830
Epoch 3/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.1885 - loss: 1.8024 - val_accuracy: 0.1684 - val_loss: 1.7747
Epoch 4/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.2036 - loss: 1.7516 - val_accuracy: 0.1850 - val_loss: 1.7697
Epoch 5/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2124 - loss: 1.7338 - val_accuracy: 0.1975 - val_loss: 1.7655
Epoch 6/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2725 - loss: 1.7088 - val_accuracy: 0.1996 - val_loss: 1.7604
Epoch 7/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2394 - loss: 1.7199 - val_accuracy: 0.2141 - val_loss: 1.7553
Epoch 8/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2887 - loss: 1.7164 - val_accuracy: 0.2225 - val_loss:

In [74]:
loss, acc = at_lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 22.50%
